# MedGemma v1.5 4B - Baseline, Ablation y LoRA

Notebook para la parte de Pablo:

1. Clonar el repo publico desde GitHub.
2. Instalar dependencias.
3. Hacer login en Hugging Face.
4. Probar MedGemma pretrained con texto e imagen.
5. Correr las 6 condiciones del pipeline mejorado.
6. Crear JSONL desde ACRIMA.
7. Arrancar un entrenamiento LoRA/QLoRA corto.

Antes de correr: cambia el runtime a GPU: `Runtime > Change runtime type > T4/L4 GPU`.

In [ ]:
# Configuracion principal
REPO_URL = "https://github.com/Luco1421/utils_medgemma.git"
REPO_DIR = "/content/utils_medgemma"

# El dataset ya viene dentro del repo publico.
LOCAL_DATASET_DIR = f"{REPO_DIR}/dataset"

MODEL_ID = "google/medgemma-1.5-4b-it"

## 1. Clonar el repo publico

In [ ]:
import os, shutil

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!ls

## 2. Instalar dependencias

Si Colab pide reiniciar runtime, reinicia y vuelve a correr desde la celda de configuracion y `%cd {REPO_DIR}`.

In [ ]:
!pip install -q -r requirements-colab.txt

## 3. Login en Hugging Face

Primero acepta los terminos del modelo en Hugging Face:

https://huggingface.co/google/medgemma-1.5-4b-it

Luego pega tu token cuando se solicite. Estar loggeado no basta si tu cuenta no acepto el acceso al modelo gated.

In [ ]:
from huggingface_hub import login
login()

## 3.1 Diagnostico de acceso a MedGemma

Esta celda debe imprimir tu usuario y luego metadata del modelo. Si falla con `GatedRepoError` o `401`, entra al link del modelo, acepta los terminos y vuelve a hacer login con un token que tenga permiso de lectura.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
print(api.whoami())
info = api.model_info(MODEL_ID, token=True)
print(info.modelId, info.private, info.gated)

## 4. Verificar dataset clonado

Como el dataset esta dentro del repo publico, no necesitamos Drive para correr las pruebas.

In [ ]:
import os
assert os.path.exists(LOCAL_DATASET_DIR), f"No existe dataset en {LOCAL_DATASET_DIR}"
!find dataset -maxdepth 2 -type f | head -20

## 5. Inspeccionar dataset ACRIMA

Esperado:

```text
dataset/Glaucoma/*.jpg
dataset/Non Glaucoma/*.jpg
```

In [ ]:
!echo "Glaucoma:" && find dataset/Glaucoma -type f | wc -l
!echo "Non Glaucoma:" && find "dataset/Non Glaucoma" -type f | wc -l

import glob
glaucoma_images = sorted(glob.glob("dataset/Glaucoma/*"))
normal_images = sorted(glob.glob("dataset/Non Glaucoma/*"))
TEST_IMAGE = glaucoma_images[0] if glaucoma_images else normal_images[0]
print("TEST_IMAGE =", TEST_IMAGE)

## 6. Preparar imports y token en el mismo kernel

A partir de aqui evitamos `!python ...`. Usamos imports para que el token y el estado de Colab vivan en el mismo proceso.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("HF token: ")

from experiments.notebook_workflows import (
    build_acrima_jsonl,
    run_ablation,
    run_smoke_inference,
    train_lora_medgemma,
)

HF_TOKEN = os.environ["HF_TOKEN"]

## 7. Smoke test: solo texto

Esto prueba que MedGemma carga y genera texto sin imagen.

In [ ]:
text_result = run_smoke_inference(
    model_id=MODEL_ID,
    hf_token=HF_TOKEN,
    prompt="Explain glaucoma in simple clinical terms.",
    output_path="results/text_only.json",
)
print(text_result["text"])

## 8. Smoke test: imagen + texto

Esto prueba el baseline multimodal pretrained.

In [ ]:
image_result = run_smoke_inference(
    model_id=MODEL_ID,
    hf_token=HF_TOKEN,
    image=TEST_IMAGE,
    prompt="Describe the ophthalmological findings in this fundus image.",
    output_path="results/image_text_baseline.json",
)
print(image_result["text"])

## 9. Ablation del pipeline mejorado

Corre A, B, C1, C2, D1, D2. Como ACRIMA no trae mascaras, usamos `--make-dummy-mask` solo para validar el flujo tecnico.

La dummy mask NO es resultado experimental; la mascara real debe venir de SAM/LoRA, WSSS o FSL/FD.

In [ ]:
ablation_result = run_ablation(
    model_id=MODEL_ID,
    hf_token=HF_TOKEN,
    image_path=TEST_IMAGE,
    prediction="glaucoma",
    distribution={"glaucoma": 0.92, "normal": 0.08},
    output_path="results/ablation_dummy.json",
)

for item in ablation_result["results"]:
    print("\n[" + item["condition"] + "]")
    print(item["text"][:1000])

## 9. Crear JSONL ACRIMA para prueba LoRA

Este JSONL se basa en etiquetas de carpeta, no en descripcion experta. Sirve para probar que el pipeline LoRA arranca.

In [ ]:
rows = build_acrima_jsonl(
    dataset_dir="dataset",
    output="data/train_medgemma_acrima.jsonl",
    max_per_class=20,
)
print(f"Ejemplos creados: {len(rows)}")
rows[:2]

## 10. Entrenamiento LoRA/QLoRA corto

Empieza con `--max-steps 5`. El objetivo inicial es validar que el entrenamiento arranca, no obtener un modelo final.

In [ ]:
adapter_dir = train_lora_medgemma(
    model_id=MODEL_ID,
    hf_token=HF_TOKEN,
    train_jsonl="data/train_medgemma_acrima.jsonl",
    image_root="dataset",
    output_dir="checkpoints/medgemma_lora_acrima_test",
    use_qlora=True,
    max_steps=5,
)
print("Adapter guardado en:", adapter_dir)

## 11. Probar adapter LoRA

In [ ]:
lora_result = run_smoke_inference(
    model_id=MODEL_ID,
    hf_token=HF_TOKEN,
    image=TEST_IMAGE,
    adapter_path="checkpoints/medgemma_lora_acrima_test",
    prompt="Describe the ophthalmological findings in this fundus image.",
    output_path="results/image_text_lora.json",
)
print(lora_result["text"])

## 12. Descargar resultados

In [ ]:
!zip -r medgemma_outputs.zip results checkpoints data || true
from google.colab import files
files.download('medgemma_outputs.zip')

## Que decir en la reunion

- El dataset actual es ACRIMA por carpetas de clase: glaucoma/no glaucoma.
- Sirve para validar inferencia baseline y LoRA tecnico.
- No trae mascaras ni descripciones expertas, por eso no reemplaza el dataset final del paper.
- La ablation A/B/C1/C2/D1/D2 ya esta implementada.
- La mascara dummy solo valida flujo; la mascara real debe venir del equipo SAM/WSSS/FSL-FD.
- LoRA queda preparado para entrenar adapters cuando haya ejemplos imagen-prompt-respuesta curados.